# Chapitre 4 — EDA diagnostique et qualité des données

**Durée estimée : 8-10 heures**

---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Distinguer** l'EDA diagnostique (trouver les problèmes) de l'EDA analytique (comprendre les patterns)
2. **Évaluer** la qualité des données selon les 6 dimensions standardisées
3. **Détecter** les valeurs manquantes, doublons et outliers avec les outils pandas appropriés
4. **Documenter** les problèmes identifiés dans une checklist de qualité priorisée

## 4.3 Statistiques descriptives de diagnostic

### Les commandes essentielles

In [2]:
import pandas as pd
import numpy as np

# Création d'un DataFrame de démonstration
np.random.seed(42) 
df = pd.DataFrame({
    'client_id': range(1, 101),
    'nom': [f'Client_{i}' for i in range(1, 101)],
    'email': [f'client{i}@test.com' if i % 10 != 0 else None for i in range(1, 101)],
    'age': np.random.randint(18, 70, 100),
    'montant_achats': np.random.uniform(100, 10000, 100).round(2)
})

# Introduisons quelques problèmes
df.loc[5, 'age'] = 150  # Âge invalide
df.loc[10, 'age'] = -5  # Âge négatif
df.loc[95:99, 'montant_achats'] = np.nan  # Valeurs manquantes

print("DataFrame de démonstration créé avec des problèmes de qualité.")
print(df)

DataFrame de démonstration créé avec des problèmes de qualité.
    client_id         nom              email  age  montant_achats
0           1    Client_1   client1@test.com   56         1736.14
1           2    Client_2   client2@test.com   69          254.80
2           3    Client_3   client3@test.com   46         4291.67
3           4    Client_4   client4@test.com   32         4009.33
4           5    Client_5   client5@test.com   60         3005.53
..        ...         ...                ...  ...             ...
95         96   Client_96  client96@test.com   42             NaN
96         97   Client_97  client97@test.com   62             NaN
97         98   Client_98  client98@test.com   58             NaN
98         99   Client_99  client99@test.com   46             NaN
99        100  Client_100               None   32             NaN

[100 rows x 5 columns]


In [3]:
# 1. Dimensions
print(f"📊 Shape : {df.shape[0]} lignes × {df.shape[1]} colonnes")

📊 Shape : 100 lignes × 5 colonnes


In [4]:
# 2. Types de données
print("📝 Types :")
print(df.dtypes)

📝 Types :
client_id           int64
nom                object
email              object
age                 int64
montant_achats    float64
dtype: object


In [5]:
# 3. Aperçu
print("👀 Premières lignes :")
df.head()

👀 Premières lignes :


,client_id,nom,email,age,montant_achats
0,1,Client_1,client1@test.com,56,1736.14
1,2,Client_2,client2@test.com,69,254.80
2,3,Client_3,client3@test.com,46,4291.67
3,4,Client_4,client4@test.com,32,4009.33
4,5,Client_5,client5@test.com,60,3005.53


In [6]:
# 4. Résumé complet
print("📋 Info :")
df.info()

📋 Info :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   client_id       100 non-null    int64  
 1   nom             100 non-null    object 
 2   email           90 non-null     object 
 3   age             100 non-null    int64  
 4   montant_achats  95 non-null     float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ KB


In [7]:
# 5. Statistiques numériques
print("📈 Describe :")
df.describe()

📈 Describe :


,client_id,age,montant_achats
count,100.000000,100.000000,95.000000
mean,50.500000,44.270000,4722.222526
std,29.011492,18.815347,2709.103418
min,1.000000,-5.000000,105.150000
25%,25.750000,32.000000,2530.455000
50%,50.500000,42.000000,4533.050000
75%,75.250000,57.000000,6780.255000
max,100.000000,150.000000,9977.630000


In [8]:
# 6. Statistiques catégorielles
print("🏷️ Valeurs uniques par colonne :")
for col in df.columns:
    print(f"{col}: {df[col].nunique()} valeurs uniques")

🏷️ Valeurs uniques par colonne :
client_id: 100 valeurs uniques
nom: 100 valeurs uniques
email: 90 valeurs uniques
age: 49 valeurs uniques
montant_achats: 95 valeurs uniques


### Tableau de référence

| Commande | Information | Utilité diagnostique |
|----------|-------------|---------------------|
| `df.shape` | (lignes, colonnes) | Volume attendu ? |
| `df.dtypes` | Types par colonne | Types corrects ? |
| `df.head()` | 5 premières lignes | Aperçu visuel |
| `df.info()` | Résumé + mémoire | Valeurs non-null |
| `df.describe()` | Stats numériques | Min/max aberrants ? |
| `df.nunique()` | Valeurs uniques | Cardinalité |
| `df.value_counts()` | Distribution | Catégories inattendues ? |

### Mesurer la complétude et l'unicité

In [ ]:
# Calculer la complétude par colonne
completude = (1 - df.isnull().mean()) * 100
# isnull().mean() calcule la proportion de valeurs manquantes par colonne
# En soustrayant de 1 et en multipliant par 100, on obtient le pourcentage de valeurs non manquantes

print("Complétude par colonne (%) :")
print(completude.sort_values())

Complétude par colonne (%) :
email              90.0
montant_achats     95.0
client_id         100.0
nom               100.0
age               100.0
dtype: float64


In [10]:
# Taux d'unicité
unicite = (1 - df.duplicated().mean()) * 100
print(f"Unicité globale : {unicite:.2f}%")

Unicité globale : 100.00%


---

### 🤖 IA : Générer du code d'exploration automatiquement

**Prompt efficace pour un LLM :**

```
J'ai un DataFrame pandas avec ces colonnes :
- client_id (int)
- nom (str)
- email (str)
- age (int)
- date_inscription (str)
- montant_achats (float)

Génère-moi un code Python complet pour :
1. Vérifier les types de données
2. Identifier les valeurs manquantes
3. Détecter les doublons potentiels
4. Chercher les outliers numériques
5. Valider les formats (email, dates)

Inclus des commentaires explicatifs.
```

---

### ✍️ Exercice 4.3 : Exploration initiale (15 min)

Analysez ce DataFrame et identifiez les problèmes potentiels :

In [1]:
import pandas as pd
import numpy as np

# Données simulées avec des problèmes
df_exercice = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 5],
    'nom': ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Eve'],
    'age': [25, 150, 35, -5, 28, 28],
    'email': ['alice@test.com', 'bob', 'charlie@test.com', None, 'eve@test.com', 'eve@test.com'],
    'salaire': [50000, 55000, np.nan, 60000, 45000, 45000],
    'date_inscription': ['2024-01-15', '2024-02-30', '2024-03-10', '2024-04-05', '2024-05-12', '2024-05-12']
})

# Votre exploration
print("1. Shape :", df_exercice.shape)
print("\n2. Types :\n", df_exercice.dtypes)
print("\n3. Valeurs manquantes :\n", df_exercice.isnull().sum())
print("\n4. Doublons :", df_exercice.duplicated().sum())
print("\n5. Stats numériques :\n", df_exercice.describe())

1. Shape : (6, 6)

2. Types :
 id                    int64
nom                  object
age                   int64
email                object
salaire             float64
date_inscription     object
dtype: object

3. Valeurs manquantes :
 id                  0
nom                 0
age                 0
email               1
salaire             1
date_inscription    0
dtype: int64

4. Doublons : 1

5. Stats numériques :
              id        age       salaire
count  6.000000    6.00000      5.000000
mean   3.333333   43.50000  51000.000000
std    1.632993   54.01759   6519.202405
min    1.000000   -5.00000  45000.000000
25%    2.250000   25.75000  45000.000000
50%    3.500000   28.00000  50000.000000
75%    4.750000   33.25000  55000.000000
max    5.000000  150.00000  60000.000000


In [ ]:
# Questions :
# a) Combien de problèmes de COMPLÉTUDE identifiez-vous ?
print("a) Problèmes de COMPLÉTUDE :")
print(f"   - email manquant : {df_exercice['email'].isnull().sum()} valeur(s)")
print(f"   - salaire manquant : {df_exercice['salaire'].isnull().sum()} valeur(s)")

# b) Combien de problèmes d'EXACTITUDE ?
print("\nb) Problèmes d'EXACTITUDE :")
print(f"   - âge > 120 : {(df_exercice['age'] > 120).sum()} valeur(s)")
print(f"   - âge < 0 : {(df_exercice['age'] < 0).sum()} valeur(s)")

# c) Combien de problèmes de VALIDITÉ ?
print("\nc) Problèmes de VALIDITÉ :")
emails_sans_at = df_exercice[~df_exercice['email'].str.contains('@', na=False)]
print(f"   - emails sans @ : {len(emails_sans_at)} valeur(s)")

# d) Combien de problèmes d'UNICITÉ ?
print("\nd) Problèmes d'UNICITÉ :")
print(f"   - doublons exacts : {df_exercice.duplicated().sum()} ligne(s)")
print(f"   - doublons sur id : {df_exercice.duplicated(subset=['id']).sum()} ligne(s)")

a) Problèmes de COMPLÉTUDE :
   - email manquant : 1 valeur(s)
   - salaire manquant : 1 valeur(s)

b) Problèmes d'EXACTITUDE :
   - âge > 120 : 1 valeur(s)
   - âge < 0 : 1 valeur(s)

c) Problèmes de VALIDITÉ :
   - emails sans @ : 2 valeur(s)

d) Problèmes d'UNICITÉ :
   - doublons exacts : 1 ligne(s)
   - doublons sur id : 1 ligne(s)
